# Theme

This project combines a real-estate property search engine with an AI expert that answers housing-related questions, helping users find suitable properties and receive professional-style real estate advice instantly.

# Install

gradio → Creates a web-based interface for property search and AI interaction.

transformers → Provides the FLAN-T5 model used to generate expert real-estate answers.

sentencepiece → Handles text tokenization so the model can understand user queries.

In [ ]:
!pip install gradio transformers sentencepiece -q

Import Libraries Cell → Loads the tools needed for data handling, UI creation, and AI response generation.

In [ ]:
import pandas as pd
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

# Load Your Real Estate Dataset
DATASET url: https://www.kaggle.com/datasets/huebitsvizg/real-state-dataset



Download the CSV file to your computer.

Open Google Drive in your browser.

Click “New” on the left side.

Select “File upload”.

Choose the CSV file from your computer.

Wait for the upload to finish.

then IT Will work

The below code ask permission to connect google drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Price Cleaning Cell → Removes commas from price values and converts them into numerical format.

In [ ]:
df = pd.read_csv("/content/realestate_synthetic_10000.csv") #Data Url taken from folder

# If price column has commas, clean it
df["price"] = df["price"].astype(str).str.replace(",", "").astype(float)

search_properties() Function Cell → Filters houses based on city, price, and bedrooms and returns only relevant listings.

Model Loading Cell → Loads the FLAN-T5 model and tokenizer to generate natural-language real estate responses.

In [ ]:
def search_properties(city, min_price, max_price, bedrooms):
    if city is None or city == "":
        return pd.DataFrame([{"Error": "Please enter a city"}])

    df_filtered = df[
        (df["city"].str.lower() == city.lower()) &
        (df["price"].between(min_price, max_price)) &
        (df["bedrooms"] == bedrooms)
    ]

    if df_filtered.empty:
        return pd.DataFrame([{"Result": "No properties found"}])

    # Return only clean important columns
    return df_filtered[[
        "property_id", "title", "city", "locality", "price", "area_sqft",
        "bedrooms", "bathrooms", "age_years", "furnishing",
        "distance_to_metro_km"
    ]]

# Load AI Model

google/flan-t5-base
is a powerful language model trained by Google that can understand text and generate clear, human-like responses, summaries, and explanations.

In [ ]:
MODEL = "google/flan-t5-base"  # fast, lightweight

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL,
    torch_dtype=torch.float32,
    device_map="cpu"
)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

# AI Assistant Function

ask_ai() Function Cell → Sends user questions to the model and returns an expert-style answer.

In [ ]:
def ask_ai(question):
    prompt = f"Answer like a real estate expert:\nQuestion: {question}\nAnswer:"

    inputs = tokenizer(prompt, return_tensors="pt")
    output = model.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7
    )
    reply = tokenizer.decode(output[0], skip_special_tokens=True)

    return reply

# Build Gradio UI

You created a tab layout:

TAB 1 → Property Search

Inputs:

City

Min price

Max price

Bedrooms

Output:

Filtered property table

TAB 2 → AI Assistant

Inputs:

Question textbox

Output:

AI-generated answer

In [ ]:
with gr.Blocks(theme=gr.themes.Soft()) as demo:

    gr.Markdown("## 🏡 AI-Powered Real Estate Assistant")

    with gr.Tabs():

        # --------------------
        # TAB 1 — Property Search
        # --------------------
        with gr.Tab("Property Search"):
            city = gr.Textbox(label="City (e.g., Mumbai)")
            min_price = gr.Number(label="Min Price", value=20000)
            max_price = gr.Number(label="Max Price", value=100000)
            bedrooms = gr.Dropdown([1,2,3,4,5], value=2, label="Bedrooms")

            search_btn = gr.Button("Search Properties 🔍")

            results = gr.DataFrame(
                interactive=False,
                wrap=True,
                label="Search Results"
            )

            search_btn.click(
                search_properties,
                inputs=[city, min_price, max_price, bedrooms],
                outputs=results
            )

        # --------------------
        # TAB 2 — AI Assistant
        # --------------------
        with gr.Tab("AI Assistant"):
            user_question = gr.Textbox(label="Ask something...")
            ask_btn = gr.Button("Ask AI 🧠")
            ai_answer = gr.Textbox(label="AI Answer")

            ask_btn.click(
                ask_ai,
                inputs=user_question,
                outputs=ai_answer
            )


# -------------------------------------------------
# 7️⃣ LAUNCH
# -------------------------------------------------
demo.launch(share=True)

/tmp/ipython-input-2264208823.py:1: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c1c3b712b070cc83f4.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
